# ATST-Frame / AST baseline for sound event detection (Vaani noise, Track 1)

Local Windows pipeline: frame-level any-noise + 7 per-category heads on top of an ATST-Frame backbone (AST fallback), BiGRU, two-stage training, decoder tuning, and `submission.zip` export.

In [2]:
"""
atst_common.py -- ATST-Frame utilities for the VAANI Track-1 SED project.

Self-contained: needs only torch + torchaudio + numpy. It deliberately does NOT import
audiossl / pytorch_lightning / fairseq / einops, so there is no fairseq stub and nothing
that breaks on Windows + Python 3.13. The encoder below is a minimal re-implementation of
audiossl's `FrameAST` (MIT licence, (c) Audio-WestlakeU) restricted to what is needed for
inference: Linear patch embedding, "cut" positional embedding, pre-LN transformer blocks with
a key-padding mask, and the final `norm_frame`. `test_atst_common.py` checks it numerically
against audiossl's own `FrameAST` when a clone of audiossl is available.

PROVENANCE (verified from the primary repositories, see PROVENANCE below):
  * `atst_as2M.ckpt` is the ATST-Frame checkpoint that ATST-SED (ICASSP'24) uses: SSL
    pre-trained, then fine-tuned on AudioSet-2M WEAK (clip-level) labels. The Google-Drive
    file id given for it in ATST-SED's README is the same id PretrainedSED lists for
    "ATST-F_weak". It is NOT `ATST-F_strong_1`, which received an extra AudioSet-Strong
    (frame-level, knowledge-distilled) training stage. Do not attribute strong-checkpoint
    results to it.

Token / time conventions (all verified in code, not assumed):
  * mel: 16 kHz, n_fft=win=1024, hop=160, 64 bands, f 60-7800 Hz, `center=True` -> mel frame m
    is centred at m*10 ms; n_frames = 1 + n_samples // 160.
  * token = 64 mel bins x 4 frames (patch_h=64, patch_w=4) -> one token per 40 ms (25 tokens/s),
    n_tokens = n_frames // 4; token p is centred at 0.04*p + 0.015 s.
  * positional table has 250 usable positions (10 s). Longer audio must be chunked.
"""
from __future__ import annotations

import hashlib
import json
import math
import os
import time
from typing import Callable, Dict, Iterable, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

# --------------------------------------------------------------------------------------
# constants
# --------------------------------------------------------------------------------------
SR = 16000
HOP = 160                                   # 10 ms mel hop
PATCH_H, PATCH_W = 64, 4                    # token = 64 mel bins x 4 frames
TOKEN_SAMPLES = PATCH_W * HOP               # 640 samples = 40 ms
TOKEN_SEC = TOKEN_SAMPLES / SR              # 0.04
TOKEN_CENTRE_SEC = (PATCH_W - 1) / 2 * (HOP / SR)   # 0.015 -> token p centre = 0.04 p + 0.015
MAX_TOKENS = 250                            # pos_embed has 1 + 250 rows (10 s)
EMBED_DIM = 768
MEL_MIN, MEL_MAX = -79.6482, 50.6842        # audiossl MinMax constants (dB)

PROVENANCE = {
    "atst_as2M.ckpt": (
        "ATST-Frame, SSL pre-trained + fine-tuned on AudioSet-2M WEAK labels "
        "(ATST-SED README: 'this checkpoint is fine-tuned by the AudioSet-2M'). Same Google-Drive "
        "id (1_xb0_n3UNbUG_pH1vLHTviLfsaSfCzxz) that PretrainedSED lists for 'ATST-F_weak'."),
    "ATST-F_strong_1.pt": (
        "ATST-F weak checkpoint + extra AudioSet-Strong training with ensemble knowledge "
        "distillation (PretrainedSED, ICASSP'25). GitHub release v0.0.1 of fschmid56/PretrainedSED. "
        "A different checkpoint with frame-level pre-training."),
    "licences": (
        "audiossl code MIT, checkpoints linked from audiossl CC BY 4.0 (audiossl LICENSE); "
        "ATST-SED code MIT; PretrainedSED code MIT (check the release page for the licence of "
        "the *_strong_1 weights before relying on them)."),
}


# --------------------------------------------------------------------------------------
# small helpers
# --------------------------------------------------------------------------------------
def sha256_file(path: str, block: int = 1 << 22) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(block)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def atst_T_expected(n_samples: int) -> int:
    """Tokens produced for an n-sample clip (mel center=True -> 1 + n//160 frames, 4 per token)."""
    return (1 + int(n_samples) // HOP) // PATCH_W


def estimate_storage_gb(total_seconds: float, dim: int = EMBED_DIM, bytes_per: int = 2, pool: int = 1) -> float:
    """ATST cache size estimate (GB) for `total_seconds` of audio, fp16 by default."""
    tokens = total_seconds / TOKEN_SEC / pool
    return tokens * dim * bytes_per / 1e9


# --------------------------------------------------------------------------------------
# preprocessing: log-mel exactly as audiossl does it (optionally the ATST-SED clamp)
# --------------------------------------------------------------------------------------
class ATSTMel(nn.Module):
    """waveform (B, N) in [-1, 1] (RAW level, no per-clip z-scoring) -> (B, 1, 64, F).

    variant "pretrain": audiossl pre-training / embedding.py pipeline (dB with top_db=80, MinMax).
    variant "sed"     : ATST-SED / PretrainedSED downstream pipeline = the above + clamp(-50, 80) dB.
    The MinMax constants are absolute dB values, so the input level matters: do NOT standardise
    the waveform per clip (that shifts every dB value by an arbitrary clip-dependent gain).
    """

    def __init__(self, variant: str = "pretrain"):
        super().__init__()
        if variant not in ("pretrain", "sed"):
            raise ValueError(variant)
        self.variant = variant
        self.mel = torchaudio.transforms.MelSpectrogram(
            SR, f_min=60, f_max=7800, hop_length=HOP, win_length=1024, n_fft=1024, n_mels=64)
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80)

    def forward(self, wav: torch.Tensor) -> torch.Tensor:
        if wav.ndim != 2:
            raise ValueError(f"expected (B, N), got {tuple(wav.shape)}")
        with torch.autocast(device_type=wav.device.type, enabled=False):
            # (B,1,N) -> (B,1,64,F): torchaudio's top_db floor is then computed PER EXAMPLE.
            # (A 3-D (B,64,F) input would take one floor over the whole batch.)
            x = self.mel(wav.float().unsqueeze(1))
            x = self.to_db(x)
            if self.variant == "sed":
                x = x.clamp(min=-50.0, max=80.0)
            x = (x - MEL_MIN) / (MEL_MAX - MEL_MIN) * 2.0 - 1.0
        return x


# --------------------------------------------------------------------------------------
# minimal FrameAST (inference only)
# --------------------------------------------------------------------------------------
class _Mlp(nn.Module):
    def __init__(self, dim: int, hidden: int):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden, dim)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


class _Attention(nn.Module):
    def __init__(self, dim: int, heads: int):
        super().__init__()
        self.heads = heads
        self.qkv = nn.Linear(dim, dim * 3, bias=False)      # qkv_bias=False in FrameAST_base
        self.proj = nn.Linear(dim, dim)

    def forward(self, x, add_mask):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.heads, C // self.heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        # softmax(q k^T / sqrt(d) + mask) v ; mask is additive (-1e4 on padded keys), as in audiossl
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=add_mask)
        return self.proj(o.transpose(1, 2).reshape(B, N, C))


class _Block(nn.Module):
    def __init__(self, dim: int, heads: int, mlp_ratio: float = 4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = _Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = _Mlp(dim, int(dim * mlp_ratio))

    def forward(self, x, add_mask):
        x = x + self.attn(self.norm1(x), add_mask)
        return x + self.mlp(self.norm2(x))


class _PatchEmbedLinear(nn.Module):
    """Key names mirror audiossl PatchEmbed_v2: `patch_embed.patch_embed.{weight,bias}`."""

    def __init__(self, dim: int):
        super().__init__()
        self.patch_embed = nn.Linear(PATCH_H * PATCH_W, dim)

    def forward(self, mel: torch.Tensor) -> torch.Tensor:
        B, _, H, Fr = mel.shape
        assert H == PATCH_H, f"mel must have {PATCH_H} bands, got {H}"
        W = Fr - Fr % PATCH_W
        # einops 'b c (h p1) (w p2) -> b (w h) (p1 p2 c)' with h=1, c=1 -> (b, w, p1*4 + p2)
        p = mel[:, 0, :, :W].reshape(B, PATCH_H, W // PATCH_W, PATCH_W).permute(0, 2, 1, 3)
        return self.patch_embed(p.reshape(B, W // PATCH_W, PATCH_H * PATCH_W))


class FrameASTMin(nn.Module):
    def __init__(self, embed_dim: int = EMBED_DIM, depth: int = 12, heads: int = 12, spec_w: int = 1001):
        super().__init__()
        self.embed_dim, self.depth = embed_dim, depth
        self.patch_embed = _PatchEmbedLinear(embed_dim)
        self.mask_embed = nn.Parameter(torch.zeros(1, 1, embed_dim))          # unused at inference, kept for strict loading
        self.pos_embed = nn.Parameter(torch.zeros(1, spec_w // PATCH_W + 1, embed_dim))
        self.blocks = nn.ModuleList([_Block(embed_dim, heads) for _ in range(depth)])
        self.norm_frame = nn.LayerNorm(embed_dim, eps=1e-6)

    @property
    def max_tokens(self) -> int:
        return self.pos_embed.shape[1] - 1

    def forward(self, mel: torch.Tensor, token_len: Optional[torch.Tensor] = None, n_last: int = 1) -> torch.Tensor:
        """mel (B,1,64,F); token_len (B,) valid tokens per item (padding is masked as keys).
        Returns the concatenation of norm_frame(x) after each of the last `n_last` blocks: (B,T,n_last*D)."""
        x = self.patch_embed(mel)
        B, T, _ = x.shape
        if T > self.max_tokens:
            raise ValueError(f"{T} tokens exceed the positional table ({self.max_tokens}); chunk the audio")
        x = x + self.pos_embed[:, 1:T + 1]
        add_mask = None
        if token_len is not None:
            pad = torch.arange(T, device=x.device)[None, :] >= token_len.to(x.device)[:, None]
            if bool(pad.any()):
                add_mask = (-10000.0 * pad[:, None, None, :].to(x.dtype))
        outs = []
        for i, blk in enumerate(self.blocks):
            x = blk(x, add_mask)
            if self.depth - i <= n_last:
                outs.append(self.norm_frame(x))
        return torch.cat(outs, dim=-1)


# --------------------------------------------------------------------------------------
# checkpoint survey + loader (ONE authoritative path)
# --------------------------------------------------------------------------------------
# (layout name, prefix) -- ordered most specific first. Mirrors the three layouts that ATST-SED's own
# `ATST.load_atst` accepts, plus the PretrainedSED wrapper layout and a bare FrameAST state dict.
LAYOUTS = (
    ("c2f_frame_encoder", "encoder.encoder.frame_encoder."),
    ("lightning_teacher", "model.teacher.encoder."),
    ("downstream_finetuned", "encoder.encoder."),
    ("pretrainedsed_wrapper", "model.atst."),
    ("bare", ""),
)


def _safe(v, depth=0):
    if isinstance(v, (int, float, bool, str)) or v is None:
        return v
    if depth > 2:
        return repr(v)[:80]
    if isinstance(v, dict):
        return {str(k): _safe(x, depth + 1) for k, x in list(v.items())[:30]}
    if isinstance(v, (list, tuple)):
        return [_safe(x, depth + 1) for x in list(v)[:30]]
    return repr(v)[:120]


def survey_checkpoint(path: str, verbose: bool = True) -> dict:
    """Open a checkpoint and report what is inside, WITHOUT building a model.
    weights_only=False is required for Lightning checkpoints (they pickle hyper-parameter objects);
    only use it on files you trust. The sha256 is recorded so the exact file is traceable."""
    if not os.path.isfile(path):
        raise FileNotFoundError(path)
    ck = torch.load(path, map_location="cpu", weights_only=False)
    info = {"path": path, "size_mb": round(os.path.getsize(path) / 1e6, 1), "sha256": sha256_file(path),
            "top_level_keys": list(ck.keys())[:20] if isinstance(ck, dict) else type(ck).__name__}
    sd = ck.get("state_dict", ck) if isinstance(ck, dict) else None
    if not isinstance(sd, dict) or not sd or not all(isinstance(k, str) for k in sd):
        raise RuntimeError(f"no usable state dict in {path}; top-level: {info['top_level_keys']}")
    sd = {k: v for k, v in sd.items() if torch.is_tensor(v)}
    info["n_tensors"] = len(sd)
    hist: Dict[str, int] = {}
    for k in sd:
        p = ".".join(k.split(".")[:3])
        hist[p] = hist.get(p, 0) + 1
    info["prefix_histogram"] = dict(sorted(hist.items(), key=lambda kv: -kv[1])[:20])
    counts = {name: sum(k.startswith(pref) for k in sd) for name, pref in LAYOUTS if pref}
    info["layout_counts"] = counts
    info["hyper_parameters"] = _safe(ck.get("hyper_parameters", {})) if isinstance(ck, dict) else {}
    pe = [k for k in sd if k.endswith("pos_embed")]
    info["pos_embed"] = {k: tuple(sd[k].shape) for k in pe}
    if verbose:
        print("[ATST] checkpoint survey")
        for k, v in info.items():
            print(f"   {k}: {v}")
    return info


def _strip(sd: Dict[str, torch.Tensor], layout: str, prefix: str) -> Dict[str, torch.Tensor]:
    out = {}
    for k, v in sd.items():
        if not k.startswith(prefix):
            continue
        k2 = k[len(prefix):]
        if k2.startswith("teacher_module.") or "cls_token" in k2:
            continue                                   # skipped by ATST-SED's loader as well
        if layout in ("lightning_teacher", "downstream_finetuned", "c2f_frame_encoder") and k2.startswith("norm."):
            k2 = "norm_frame." + k2[len("norm."):]      # ATST-SED: encoder.norm -> norm_frame
        out[k2] = v
    return out


def remap_state_dict(sd: Dict[str, torch.Tensor]) -> Tuple[Dict[str, torch.Tensor], str]:
    """Pick the checkpoint layout by key counts and return the encoder-only state dict."""
    sd = {k: v for k, v in sd.items() if torch.is_tensor(v)}
    counts = {name: sum(k.startswith(pref) for k in sd) for name, pref in LAYOUTS if pref}
    min_keys = 100                                      # a 12-block encoder has ~150 tensors
    cand = [(n, p) for n, p in LAYOUTS if p and counts[n] >= min_keys]
    if "c2f_frame_encoder" in [c[0] for c in cand]:
        cand = [c for c in cand if c[0] != "downstream_finetuned"]   # superset prefix: prefer the specific one
    if len(cand) > 1:
        raise RuntimeError(f"ambiguous checkpoint layout, key counts {counts}. Refusing to guess "
                           f"(student/teacher style checkpoints must be resolved explicitly).")
    if cand:
        name, prefix = cand[0]
    elif any(k.startswith("blocks.0.") for k in sd):
        name, prefix = "bare", ""
    else:
        raise RuntimeError(f"unrecognised checkpoint layout; key counts {counts}; sample keys: {list(sd)[:8]}")
    return _strip(sd, name, prefix), name


class ATSTEncoder(nn.Module):
    """mel front-end + FrameASTMin + provenance metadata."""

    def __init__(self, net: FrameASTMin, mel: ATSTMel, meta: dict):
        super().__init__()
        self.net, self.mel, self.meta = net, mel, meta


def load_atst_encoder(ckpt_path: str, device="cpu", mel_variant: str = "pretrain", freeze: bool = True,
                      expected_sha256: Optional[str] = None, verbose: bool = True) -> ATSTEncoder:
    """The single ATST loading path. Fails loudly: strict key matching, no silent fallback of any kind
    (no AST substitute, no randomly initialised layers). Returns an eval-mode, frozen encoder."""
    info = survey_checkpoint(ckpt_path, verbose=verbose)
    if expected_sha256 and info["sha256"] != expected_sha256:
        raise RuntimeError(f"sha256 mismatch: {info['sha256']} != {expected_sha256}")
    raw = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    sd = raw.get("state_dict", raw)
    enc_sd, layout = remap_state_dict(sd)
    if any(".prompt_embed" in k or k == "prompt_embed" for k in enc_sd):
        raise NotImplementedError("checkpoint uses prompt tokens (nprompt>0); not supported here")
    if any(k.startswith("patch_embed.proj.") for k in enc_sd):
        raise NotImplementedError("checkpoint uses the CNN patch embedding; only the Linear one is implemented")
    try:
        dim = int(enc_sd["pos_embed"].shape[-1]); spec_w = (int(enc_sd["pos_embed"].shape[1]) - 1) * PATCH_W
        depth = 1 + max(int(k.split(".")[1]) for k in enc_sd if k.startswith("blocks."))
    except KeyError as e:
        raise RuntimeError(f"encoder state dict lacks {e}; layout={layout}; keys: {list(enc_sd)[:8]}")
    heads = {768: 12, 384: 6}.get(dim)
    if heads is None:
        raise NotImplementedError(f"embed dim {dim}")
    net = FrameASTMin(embed_dim=dim, depth=depth, heads=heads, spec_w=spec_w)
    try:
        net.load_state_dict(enc_sd, strict=True)
    except RuntimeError as e:
        raise RuntimeError(f"strict load failed (layout={layout}): {str(e)[:1200]}") from e
    mel = ATSTMel(mel_variant)
    n_params = sum(p.numel() for p in net.parameters())
    meta = {"ckpt": os.path.abspath(ckpt_path), "sha256": info["sha256"], "layout": layout, "embed_dim": dim,
            "depth": depth, "n_params": n_params, "mel_variant": mel_variant,
            "token_sec": TOKEN_SEC, "max_tokens": net.max_tokens,
            "provenance": "ATST-Frame, weak-label (AudioSet-2M) fine-tuned unless the sha256 says otherwise"}
    enc = ATSTEncoder(net, mel, meta).to(device).eval()
    if freeze:
        for p in enc.parameters():
            p.requires_grad_(False)
    if verbose:
        print(f"[ATST] loaded OK: layout={layout} dim={dim} depth={depth} params={n_params / 1e6:.1f}M "
              f"mel={mel_variant} sha256={info['sha256'][:16]}...")
    return enc


# --------------------------------------------------------------------------------------
# clip -> token features (chunked for audio longer than 10 s)
# --------------------------------------------------------------------------------------
def plan_chunks(n_tok: int, chunk: int = MAX_TOKENS, hop: int = 125) -> List[Tuple[int, int, int, int]]:
    """[(start, end, keep_start, keep_end)] in token units. Keep ranges tile [0, n_tok) exactly; the
    boundary between neighbouring chunks is the midpoint of their overlap, so each kept token has
    at least ~(chunk-hop)/2 tokens of context on each side (except at the clip edges)."""
    if n_tok <= 0:
        return []
    if n_tok <= chunk:
        return [(0, n_tok, 0, n_tok)]
    if hop >= chunk:
        raise ValueError("hop must be < chunk")
    starts = list(range(0, n_tok - chunk + 1, hop))
    if starts[-1] + chunk < n_tok:
        starts.append(n_tok - chunk)
    plan = []
    for i, s in enumerate(starts):
        e = s + chunk
        k0 = 0 if i == 0 else (s + starts[i - 1] + chunk) // 2
        k1 = n_tok if i == len(starts) - 1 else (e + starts[i + 1]) // 2
        plan.append((s, e, k0, k1))
    return plan


class ATSTFeatureExtractor:
    """clip waveform -> (T, D) float32 token features on the 40 ms grid.
    The mel is computed ONCE for the whole clip (so chunk seams never see reflection padding) and then
    sliced into <=250-token chunks. Clips shorter than 1 s are zero-padded to 1 s for extraction only,
    exactly like V5's test-BEATs extraction; the returned T is atst_T_expected(max(n, SR))."""

    def __init__(self, enc: ATSTEncoder, device, n_last: int = 1, chunk: int = MAX_TOKENS, hop: int = 125,
                 batch_chunks: int = 8):
        self.enc, self.device, self.n_last = enc, torch.device(device), n_last
        self.chunk, self.hop, self.batch_chunks = chunk, hop, batch_chunks
        self.dim = enc.net.embed_dim

    @staticmethod
    def padded_len(n_samples: int) -> int:
        return max(int(n_samples), SR)

    @torch.no_grad()
    def __call__(self, wav) -> np.ndarray:
        w = torch.as_tensor(wav, dtype=torch.float32).flatten()
        if w.numel() < SR:
            w = F.pad(w, (0, SR - w.numel()))
        w = w.to(self.device)
        mel = self.enc.mel(w[None])                                  # (1,1,64,F)
        n_tok = mel.shape[-1] // PATCH_W
        assert n_tok == atst_T_expected(w.numel()), (n_tok, atst_T_expected(w.numel()))
        plan = plan_chunks(n_tok, self.chunk, self.hop)
        out = torch.zeros(n_tok, self.dim * self.n_last, device=self.device)
        for b0 in range(0, len(plan), self.batch_chunks):
            part = plan[b0:b0 + self.batch_chunks]
            width = max(e - s for s, e, _, _ in part) * PATCH_W
            fill = float(mel.min())
            batch = torch.stack([F.pad(mel[0, :, :, s * PATCH_W:e * PATCH_W], (0, width - (e - s) * PATCH_W), value=fill)
                                 for s, e, _, _ in part])            # (b,1,64,width)
            lens = torch.tensor([e - s for s, e, _, _ in part], device=self.device)
            feats = self.enc.net(batch, lens, n_last=self.n_last)    # (b,T,n_last*D)
            for j, (s, e, k0, k1) in enumerate(part):
                out[k0:k1] = feats[j, k0 - s:k1 - s]
        if self.n_last > 1:
            out = out.reshape(n_tok, self.n_last, self.dim).mean(1)
        return out.float().cpu().numpy()


# --------------------------------------------------------------------------------------
# alignment of cached token rows to V5 windows
# --------------------------------------------------------------------------------------
def atst_rows_for_window(atst_full: np.ndarray, start_sample: int, canon_t: int = 200) -> Tuple[np.ndarray, int]:
    """Token rows for an 8 s window starting at `start_sample` (must sit on the 0.16 s V5 patch grid, which
    is 4 tokens). Returns (rows[canon_t, D], n_real_rows). Zero-pads past the end of the clip."""
    if start_sample % TOKEN_SAMPLES:
        raise AssertionError(f"window start {start_sample} is not on the 40 ms token grid")
    k0 = start_sample // TOKEN_SAMPLES
    rows = atst_full[k0:k0 + canon_t]
    n_real = rows.shape[0]
    if n_real < canon_t:
        rows = np.concatenate([rows, np.zeros((canon_t - n_real, atst_full.shape[1]), atst_full.dtype)], 0)
    return rows, n_real


def atst_rows_for_shifted_clip(atst_full: np.ndarray, shift_tokens: int, canon_t: int = 200) -> Tuple[np.ndarray, int]:
    """Whole clip placed `shift_tokens` tokens into an 8 s window (V5 shift augmentation: k patches = 4k tokens)."""
    D = atst_full.shape[1]
    body = atst_full[:max(0, canon_t - shift_tokens)]
    n_real = body.shape[0]
    rows = np.concatenate([np.zeros((shift_tokens, D), atst_full.dtype), body,
                           np.zeros((canon_t - shift_tokens - n_real, D), atst_full.dtype)], 0)
    return rows, n_real


def interp_landing_time(p: int, n_rows: int = 200, w_sec: float = 8.0) -> float:
    """Time (s, within the window) at which F.interpolate(align_corners=False) puts input row p on the
    V5 grid: (p + 0.5) * w_sec / n_rows. ATST: 0.04p + 0.02 vs true token centre 0.04p + 0.015 (5 ms)."""
    return (p + 0.5) * w_sec / n_rows


# --------------------------------------------------------------------------------------
# resumable, verified feature cache
# --------------------------------------------------------------------------------------
class ATSTCache:
    """root/<kind>/<key>.npy (fp16, (T,D)); root/meta.json pins checkpoint sha256 + preprocessing.
    kinds: 'nat' (global dataset index), 'synth' (synthetic id), 'test' (clip id)."""

    def __init__(self, root: str, tag: str, meta: dict, create: bool = True):
        self.root, self.tag = root, tag
        self.meta = dict(meta, tag=tag)
        if create:
            os.makedirs(root, exist_ok=True)
        mp = os.path.join(root, "meta.json")
        if os.path.exists(mp):
            with open(mp) as f:
                old = json.load(f)
            for k in ("tag", "sha256", "mel_variant", "n_last", "chunk", "hop", "embed_dim"):
                if old.get(k) != self.meta.get(k):
                    raise RuntimeError(f"cache {root} was built with {k}={old.get(k)!r}, now {self.meta.get(k)!r}. "
                                       f"Use a new tag/folder instead of mixing features.")
        elif create:
            with open(mp, "w") as f:
                json.dump(dict(self.meta, created=time.strftime("%Y-%m-%d %H:%M:%S")), f, indent=1)

    @staticmethod
    def _name(key) -> str:
        return f"{int(key):06d}" if isinstance(key, (int, np.integer)) else str(key)

    def path(self, kind: str, key) -> str:
        return os.path.join(self.root, kind, self._name(key) + ".npy")

    def has(self, kind: str, key) -> bool:
        return os.path.exists(self.path(kind, key))

    def load(self, kind: str, key) -> np.ndarray:
        p = self.path(kind, key)
        if not os.path.exists(p):
            raise RuntimeError(f"missing ATST cache entry {p} (no live fallback by design)")
        return np.load(p)

    def save(self, kind: str, key, arr16: np.ndarray):
        p = self.path(kind, key)
        os.makedirs(os.path.dirname(p), exist_ok=True)
        tmp = p + ".tmp.npy"
        np.save(tmp, arr16)
        os.replace(tmp, p)


def build_cache(items: Iterable, get_wav: Callable, cache: ATSTCache, kind: str, extractor: ATSTFeatureExtractor,
                key_of: Callable = lambda x: x, log: Callable = print, max_new: Optional[int] = None,
                fp16_check_every: int = 200, log_every: int = 500) -> dict:
    """Extract + store missing entries. Hard-fails on non-finite output, wrong shape or fp16 overflow.
    Resumable (existing files are skipped; use verify_cache to audit them)."""
    t0 = time.time(); n_new = n_skip = 0; worst_cos = 1.0; worst_abs = 0.0
    for i, it in enumerate(items):
        key = key_of(it)
        if cache.has(kind, key):
            n_skip += 1
            continue
        if max_new is not None and n_new >= max_new:
            break
        w = torch.as_tensor(get_wav(it), dtype=torch.float32).flatten()
        feat = extractor(w)                                         # fp32 (T,D)
        T_exp = atst_T_expected(extractor.padded_len(w.numel()))
        if feat.shape != (T_exp, extractor.dim):
            raise RuntimeError(f"{kind}:{key}: shape {feat.shape}, expected {(T_exp, extractor.dim)}")
        if not np.isfinite(feat).all():
            raise RuntimeError(f"{kind}:{key}: non-finite ATST features")
        if np.abs(feat).max() > 6e4:
            raise RuntimeError(f"{kind}:{key}: |feature| {np.abs(feat).max():.0f} would overflow fp16")
        f16 = feat.astype(np.float16)
        if fp16_check_every and n_new % fp16_check_every == 0:
            a, b = feat.reshape(-1), f16.astype(np.float32).reshape(-1)
            worst_cos = min(worst_cos, float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)))
            worst_abs = max(worst_abs, float(np.abs(a - b).max()))
        cache.save(kind, key, f16); n_new += 1
        if n_new % log_every == 0:
            log(f"  [atst-cache:{kind}] {n_new} new, {n_skip} skipped, {time.time() - t0:.0f}s")
    stats = {"kind": kind, "new": n_new, "skipped": n_skip, "seconds": round(time.time() - t0, 1),
             "fp16_min_cosine": worst_cos, "fp16_max_abs_err": worst_abs}
    if worst_cos < 0.9999:
        raise RuntimeError(f"fp16 storage too lossy: cosine {worst_cos:.6f}")
    log(f"  [atst-cache:{kind}] done {stats}")
    return stats


def verify_cache(cache: ATSTCache, kind: str, keys: Iterable, n_samples_of: Callable, log: Callable = print,
                 full: bool = False) -> dict:
    """Exhaustive presence check (+ shape/finite when full=True). Raises on any problem."""
    missing, bad = [], []
    keys = list(keys)
    for k in keys:
        if not cache.has(kind, k):
            missing.append(k); continue
        if full:
            a = cache.load(kind, k)
            exp = atst_T_expected(max(int(n_samples_of(k)), SR))
            if a.ndim != 2 or a.shape[0] != exp or not np.isfinite(a.astype(np.float32)).all():
                bad.append((k, a.shape, exp))
    if missing or bad:
        raise RuntimeError(f"[{kind}] ATST cache problems: {len(missing)} missing (first {missing[:5]}), "
                           f"{len(bad)} bad (first {bad[:3]})")
    log(f"  [atst-cache:{kind}] verified {len(keys)} entries" + (" (shape+finite)" if full else " (presence)"))
    return {"kind": kind, "n": len(keys)}
    

In [3]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB"
)

Python: C:\Users\231b258\torch_gpu\Scripts\python.exe
PyTorch: 2.14.0+cu132
CUDA: 13.2
CUDA available: True
GPU: NVIDIA RTX PRO 2000 Blackwell
VRAM: 15.93 GB


In [4]:
# =============================================================================
# Stub fairseq -- only used by audiossl for pretraining-time masking.
# During fine-tuning/inference the masking function is never called, but the
# import at the top of random_mask.py fails if fairseq isn't installed.
# fairseq itself won't build on Windows/Python 3.13, so we inject a stub.
# =============================================================================
import types
if 'fairseq' not in sys.modules:
    _fs = types.ModuleType('fairseq')
    _fs_data = types.ModuleType('fairseq.data')
    _fs_utils = types.ModuleType('fairseq.data.data_utils')

    def _compute_mask_indices(shape, padding_mask=None, mask_prob=0.65,
                              mask_length=10, mask_type="static", **kwargs):
        # Returns bool array (False = don't mask). We never mask during
        # inference, so an all-False array is correct here.
        import numpy as _np
        return _np.zeros(shape, dtype=bool)

    _fs_utils.compute_mask_indices = _compute_mask_indices
    _fs_data.data_utils = _fs_utils
    _fs.data = _fs_data

    sys.modules['fairseq'] = _fs
    sys.modules['fairseq.data'] = _fs_data
    sys.modules['fairseq.data.data_utils'] = _fs_utils
    print("[stub] fairseq.data.data_utils.compute_mask_indices installed")

[stub] fairseq.data.data_utils.compute_mask_indices installed


## Cell 1 — Environment setup

Run once. Assumes a Python 3.10+ environment with CUDA-capable PyTorch already installed. The three packages below are the only additions needed.

In [5]:
# =============================================================================
# CELL 1 -- imports, paths, ATST-Frame teacher encoder loader
# =============================================================================

import sys
import types
import os
import io
import json
import math
import random
import time
import zipfile
import tempfile
from pathlib import Path
from collections import Counter

# -----------------------------------------------------------------------------
# 1. Make cloned audiossl repo importable
# -----------------------------------------------------------------------------
_AUDIOSS_PATH = r"C:/Users/231b258/audiossl"

if _AUDIOSS_PATH not in sys.path:
    sys.path.insert(0, _AUDIOSS_PATH)

print(f"[audiossl] source = {_AUDIOSS_PATH}")

# -----------------------------------------------------------------------------
# 2. fairseq compatibility stub
#
# audiossl imports compute_mask_indices during module import.
# We don't use masking for downstream embedding extraction.
# -----------------------------------------------------------------------------
if "fairseq" not in sys.modules:

    _fs = types.ModuleType("fairseq")
    _fs_data = types.ModuleType("fairseq.data")
    _fs_utils = types.ModuleType("fairseq.data.data_utils")

    def _compute_mask_indices(
        shape,
        padding_mask=None,
        mask_prob=0.65,
        mask_length=10,
        mask_type="static",
        **kwargs,
    ):
        import numpy as _np
        return _np.zeros(shape, dtype=bool)

    _fs_utils.compute_mask_indices = _compute_mask_indices
    _fs_data.data_utils = _fs_utils
    _fs.data = _fs_data

    sys.modules["fairseq"] = _fs
    sys.modules["fairseq.data"] = _fs_data
    sys.modules["fairseq.data.data_utils"] = _fs_utils

    print("[fairseq] compatibility stub installed")

# -----------------------------------------------------------------------------
# 3. Standard imports
# -----------------------------------------------------------------------------
import numpy as np
import soundfile as sf
import librosa

from scipy.ndimage import median_filter

# -----------------------------------------------------------------------------
# 4. PyTorch
# -----------------------------------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

print(
    f"[torch] version={torch.__version__} | "
    f"cuda={torch.cuda.is_available()} | "
    f"device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}"
)

# -----------------------------------------------------------------------------
# 5. HuggingFace datasets
# -----------------------------------------------------------------------------
from datasets import load_from_disk

# -----------------------------------------------------------------------------
# 6. torchvision transforms
# -----------------------------------------------------------------------------
from torchvision import transforms

# -----------------------------------------------------------------------------
# 7. Import ATST-Frame components
#
# IMPORTANT:
# Do NOT use audiossl.embedding.load_model().
# That loader reconstructs the complete teacher+student Lightning module,
# and your checkpoint does not need the student branch for downstream SED.
# -----------------------------------------------------------------------------
HAS_ATST = False

try:
    from audiossl.methods.atstframe.audio_transformer import (
        FrameAST_small,
        FrameAST_base,
    )

    from audiossl.methods.atstframe import embedding as _atst_embedding

    get_timestamp_embedding = _atst_embedding.get_timestamp_embedding

    # Exact preprocessing objects from audiossl
    melspec_t = _atst_embedding.melspec_t
    to_db = _atst_embedding.to_db
    normalize = _atst_embedding.normalize

    HAS_ATST = True

    print("[ATST] ATST-Frame modules imported successfully")

except Exception as e:
    print(f"[ATST] import failed: {type(e).__name__}: {e}")

# -----------------------------------------------------------------------------
# 8. ATST checkpoint
# -----------------------------------------------------------------------------
ATST_CKPT = r"C:/Users/231b258/atst_as2M.ckpt"

# -----------------------------------------------------------------------------
# 9. Direct ATST teacher encoder loader
# -----------------------------------------------------------------------------
def load_atstframe(model_path):
    """
    Load only the ATST-Frame teacher encoder from atst_as2M.ckpt.

    This avoids reconstructing the Lightning student branch.

    The loader:
      1. loads the old Lightning checkpoint with weights_only=False
      2. extracts model.teacher.encoder.*
      3. infers Base vs Small from the checkpoint tensor dimensions
      4. constructs the correct FrameAST architecture
      5. loads only teacher encoder parameters
      6. attaches the official audiossl preprocessing transform
    """

    if not HAS_ATST:
        raise RuntimeError(
            "ATST modules were not imported. "
            "Check the [ATST] import error above."
        )

    model_path = str(model_path)

    if not os.path.isfile(model_path):
        raise FileNotFoundError(
            f"ATST checkpoint not found:\n{model_path}"
        )

    print("=" * 70)
    print("[ATST] Loading checkpoint")
    print("=" * 70)
    print(model_path)

    # -------------------------------------------------------------------------
    # PyTorch 2.6+ compatibility.
    # This is a trusted local checkpoint.
    # -------------------------------------------------------------------------
    checkpoint = torch.load(
        model_path,
        map_location="cpu",
        weights_only=False,
    )

    if not isinstance(checkpoint, dict):
        raise RuntimeError(
            "Checkpoint is not a dictionary/Lightning checkpoint."
        )

    if "state_dict" not in checkpoint:
        raise RuntimeError(
            "Checkpoint does not contain a 'state_dict'."
        )

    state_dict = checkpoint["state_dict"]
    hyper_parameters = checkpoint.get("hyper_parameters", {})

    print(f"[ATST] checkpoint keys       : {len(state_dict):,}")
    print(
        f"[ATST] hyperparameters       : "
        f"{list(hyper_parameters.keys())}"
    )

    # -------------------------------------------------------------------------
    # Find the teacher encoder prefix.
    #
    # Official audiossl checkpoints use:
    #     model.teacher.encoder.*
    # -------------------------------------------------------------------------
    possible_prefixes = [
        "model.teacher.encoder.",
        "teacher.encoder.",
    ]

    teacher_prefix = None

    for prefix in possible_prefixes:
        if any(k.startswith(prefix) for k in state_dict.keys()):
            teacher_prefix = prefix
            break

    if teacher_prefix is None:

        # Print useful diagnostics before failing.
        sample_keys = list(state_dict.keys())[:30]

        raise RuntimeError(
            "Could not find teacher encoder weights in checkpoint.\n\n"
            "First checkpoint keys:\n" +
            "\n".join(f"  {k}" for k in sample_keys)
        )

    print(f"[ATST] teacher prefix        : {teacher_prefix}")

    # -------------------------------------------------------------------------
    # Extract only teacher encoder weights and remove prefix.
    # -------------------------------------------------------------------------
    teacher_state = {
        k[len(teacher_prefix):]: v
        for k, v in state_dict.items()
        if k.startswith(teacher_prefix)
    }

    print(
        f"[ATST] teacher encoder keys  : "
        f"{len(teacher_state):,}"
    )

    if not teacher_state:
        raise RuntimeError(
            "Teacher encoder state_dict is empty."
        )

    # -------------------------------------------------------------------------
    # Infer Base vs Small from an attention weight.
    #
    # Base:
    #   embed_dim = 768
    #   qkv.weight shape = [2304, 768]
    #
    # Small:
    #   embed_dim = 384
    #   qkv.weight shape = [1152, 384]
    # -------------------------------------------------------------------------
    qkv_key = None

    for k in teacher_state:
        if k.endswith("blocks.0.attn.qkv.weight"):
            qkv_key = k
            break

    if qkv_key is None:
        raise RuntimeError(
            "Could not find blocks.0.attn.qkv.weight "
            "inside teacher encoder checkpoint."
        )

    qkv_shape = tuple(teacher_state[qkv_key].shape)

    if qkv_shape[1] == 768:
        arch = "base"
        encoder_dim = 768
        print(
            f"[ATST] architecture detected: BASE "
            f"(768-dim), qkv={qkv_shape}"
        )

    elif qkv_shape[1] == 384:
        arch = "small"
        encoder_dim = 384
        print(
            f"[ATST] architecture detected: SMALL "
            f"(384-dim), qkv={qkv_shape}"
        )

    else:
        raise RuntimeError(
            f"Unknown ATST architecture. "
            f"qkv weight shape = {qkv_shape}"
        )

    # -------------------------------------------------------------------------
    # Read compatible architecture parameters.
    # -------------------------------------------------------------------------
    pos_type = hyper_parameters.get("pos_type", "cut")
    patch_embed = hyper_parameters.get("patch_embed", "Linear")
    nprompt = int(hyper_parameters.get("nprompt", 0))

    # -------------------------------------------------------------------------
    # Construct ONLY the encoder.
    #
    # This completely avoids the student branch.
    # -------------------------------------------------------------------------
    if arch == "base":

        encoder = FrameAST_base(
            patch_h=64,
            patch_w=4,
            pos_type=pos_type,
            patch_embed=patch_embed,
            nprompt=nprompt,
        )

    else:

        encoder = FrameAST_small(
            patch_h=64,
            patch_w=4,
            pos_type=pos_type,
            patch_embed=patch_embed,
            nprompt=nprompt,
        )

    # -------------------------------------------------------------------------
    # Load teacher encoder weights.
    #
    # strict=False is used only as a compatibility guard.
    # We explicitly check the remaining missing/unexpected parameters.
    # -------------------------------------------------------------------------
    missing_keys, unexpected_keys = encoder.load_state_dict(
        teacher_state,
        strict=False,
    )

    print(f"[ATST] missing encoder keys : {len(missing_keys)}")
    print(f"[ATST] unexpected keys       : {len(unexpected_keys)}")

    # These should normally both be zero for the correct checkpoint.
    if missing_keys:
        print("[ATST] Missing keys:")
        for k in missing_keys[:30]:
            print(f"    {k}")

    if unexpected_keys:
        print("[ATST] Unexpected keys:")
        for k in unexpected_keys[:30]:
            print(f"    {k}")

    # -------------------------------------------------------------------------
    # Hard safety check.
    #
    # If actual encoder parameters are missing, don't silently train with
    # randomly initialized backbone layers.
    # -------------------------------------------------------------------------
    if len(missing_keys) > 0:
        raise RuntimeError(
            f"ATST teacher encoder has {len(missing_keys)} missing parameters. "
            "The checkpoint and audiossl encoder definition are incompatible."
        )

    # -------------------------------------------------------------------------
    # Metadata expected by downstream audiossl code.
    # -------------------------------------------------------------------------
    N_BLOCKS = 12

    encoder.hyper_param = hyper_parameters
    encoder.sample_rate = 16000

    encoder.scene_embedding_size = (
        encoder_dim * 2 * N_BLOCKS
    )

    encoder.timestamp_embedding_size = (
        encoder_dim * N_BLOCKS
    )

    # -------------------------------------------------------------------------
    # IMPORTANT:
    #
    # Do NOT first assign nn.Sequential().
    # torchvision.transforms.Compose is NOT an nn.Module.
    #
    # The original source expects:
    #     model.transform.transforms[0]
    #
    # so use torchvision Compose exactly as upstream does.
    # -------------------------------------------------------------------------
    encoder.transform = transforms.Compose([
        melspec_t,
        to_db,
        normalize,
    ])

    # -------------------------------------------------------------------------
    # Freeze backbone
    # -------------------------------------------------------------------------
    encoder.eval()

    for p in encoder.parameters():
        p.requires_grad_(False)

    # -------------------------------------------------------------------------
    # Final diagnostics
    # -------------------------------------------------------------------------
    n_params = sum(
        p.numel()
        for p in encoder.parameters()
    )

    print("=" * 70)
    print("[ATST] TEACHER ENCODER READY")
    print("=" * 70)
    print(f"architecture       : {arch}")
    print(f"embedding dimension : {encoder_dim}")
    print(f"parameters          : {n_params:,}")
    print(f"timestamp dimension : {encoder.timestamp_embedding_size}")
    print(f"scene dimension     : {encoder.scene_embedding_size}")
    print("=" * 70)

    return encoder


# -----------------------------------------------------------------------------
# 10. Final environment summary
# -----------------------------------------------------------------------------
print("=" * 70)
print("ENVIRONMENT")
print("=" * 70)
print(f"HAS_ATST={HAS_ATST}")
print(f"ATST_CKPT={ATST_CKPT}")
print("=" * 70)

[audiossl] source = C:/Users/231b258/audiossl
[torch] version=2.14.0+cu132 | cuda=True | device=NVIDIA RTX PRO 2000 Blackwell


C:\Users\231b258\torch_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1010 11:26:22.250000 22732 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


[ATST] ATST-Frame modules imported successfully
ENVIRONMENT
HAS_ATST=True
ATST_CKPT=C:/Users/231b258/atst_as2M.ckpt


In [6]:
# =============================================================================
# CELL 2 -- verify ATST checkpoint and timestamp embeddings
# =============================================================================

import torch

atst = load_atstframe(ATST_CKPT)

# 2 x 10 seconds of fake 16-kHz audio
x = torch.randn(2, 160000)

with torch.no_grad():
    emb, timestamps = get_timestamp_embedding(x, atst)

print("=" * 70)
print("ATST TEST")
print("=" * 70)
print("embedding shape :", tuple(emb.shape))
print("timestamp shape :", tuple(timestamps.shape))
print("dtype           :", emb.dtype)
print("NaN             :", torch.isnan(emb).any().item())
print("Inf             :", torch.isinf(emb).any().item())
print("=" * 70)

[ATST] Loading checkpoint
C:/Users/231b258/atst_as2M.ckpt
[ATST] checkpoint keys       : 143
[ATST] hyperparameters       : ['learning_rate', 'max_epochs', 'niter_per_epoch', 'warmup_epochs', 'num_labels', 'multi_label', 'layer_wise_lr', 'freeze_embed', 'mixup_training', 'optimizer', 'n_last_blocks', 'pretrained_ckpt_path', 'save_path', 'layerwise_lr', 'mask_aug', 'rrc', 'nproc', 'mixup_ratio', 'alpha', 'roll_mag', 'dataset_name', 'data_path', 'batch_size_per_gpu', 'num_workers']


RuntimeError: Could not find teacher encoder weights in checkpoint.

First checkpoint keys:
  encoder.encoder.mask_embed
  encoder.encoder.pos_embed
  encoder.encoder.patch_embed.patch_embed.weight
  encoder.encoder.patch_embed.patch_embed.bias
  encoder.encoder.blocks.0.norm1.weight
  encoder.encoder.blocks.0.norm1.bias
  encoder.encoder.blocks.0.attn.qkv.weight
  encoder.encoder.blocks.0.attn.proj.weight
  encoder.encoder.blocks.0.attn.proj.bias
  encoder.encoder.blocks.0.norm2.weight
  encoder.encoder.blocks.0.norm2.bias
  encoder.encoder.blocks.0.mlp.fc1.weight
  encoder.encoder.blocks.0.mlp.fc1.bias
  encoder.encoder.blocks.0.mlp.fc2.weight
  encoder.encoder.blocks.0.mlp.fc2.bias
  encoder.encoder.blocks.1.norm1.weight
  encoder.encoder.blocks.1.norm1.bias
  encoder.encoder.blocks.1.attn.qkv.weight
  encoder.encoder.blocks.1.attn.proj.weight
  encoder.encoder.blocks.1.attn.proj.bias
  encoder.encoder.blocks.1.norm2.weight
  encoder.encoder.blocks.1.norm2.bias
  encoder.encoder.blo

In [ ]:
import os, io, json, math, random, time, zipfile, tempfile
from pathlib import Path
from collections import Counter

import numpy as np
import soundfile as sf
import librosa
from scipy.ndimage import median_filter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

from datasets import load_from_disk

# --- ATST-Frame loader (audiossl) ---
try:
    from audiossl.methods.atstframe.embedding import load_model as load_atstframe
    HAS_ATST = True
    print("[ATST] audiossl available")
except Exception as e:
    HAS_ATST = False
    print(f"[ATST] audiossl not available ({type(e).__name__}); will fall back to AST")

print(f"torch={torch.__version__}  cuda={torch.cuda.is_available()}  "
      f"device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

## Cell 2 — Paths and directory structure

Everything lives under `BASE_DIR/atst_baseline/`. Nothing here overlaps with your VaaniLiteSED pipeline (`beats_cache_full/`, `wav_cache_16k_flac/`, `best_stage*.pt`).

In [ ]:
# ---- Local paths (edit BASE_DIR if you relocate) ----
if "BASE_DIR" not in dir():
    BASE_DIR = "C:/Users/231b258"

ATST_WORK_DIR    = f"{BASE_DIR}/atst_baseline"
ATST_CKPT_DIR    = f"{ATST_WORK_DIR}/checkpoints"
ATST_LOG_DIR     = f"{ATST_WORK_DIR}/logs"
ATST_SUB_DIR     = f"{ATST_WORK_DIR}/submission"
ATST_MODEL_CACHE = f"{ATST_WORK_DIR}/model_cache"
ATST_TRACKS_DIR  = f"{ATST_WORK_DIR}/tracks"
for _d in (ATST_WORK_DIR, ATST_CKPT_DIR, ATST_LOG_DIR,
           ATST_SUB_DIR, ATST_MODEL_CACHE, ATST_TRACKS_DIR):
    os.makedirs(_d, exist_ok=True)

ATST_PROGRESS_LOG  = f"{ATST_LOG_DIR}/progress_log.txt"
ATST_PROGRESS_LIVE = f"{ATST_LOG_DIR}/progress_live.txt"

# ---- Dataset & test audio (reuse the local copies the other pipeline uses) --
if "DATASET_PATH" not in dir():
    _root = f"{BASE_DIR}/archive/vaani_noise_dataset"
    DATASET_PATH = f"{_root}/train" if os.path.isdir(f"{_root}/train") else _root
if "TEST_AUDIO_DIR" not in dir():
    TEST_AUDIO_DIR = f"{BASE_DIR}/test/audio"

assert os.path.isdir(DATASET_PATH), f"Dataset not found: {DATASET_PATH}"
assert os.path.isdir(TEST_AUDIO_DIR), f"Test dir not found: {TEST_AUDIO_DIR}"

# ---- ATST-Frame checkpoint (actual file at BASE_DIR root) ----
ATST_FRAME_CKPT = f"{BASE_DIR}/atst_as2M.ckpt"
HAS_ATST_FRAME = HAS_ATST and os.path.isfile(ATST_FRAME_CKPT)

print(f"work dir     : {ATST_WORK_DIR}")
print(f"  ckpt       : {ATST_CKPT_DIR}")
print(f"  logs       : {ATST_LOG_DIR}")
print(f"  submission : {ATST_SUB_DIR}")
print(f"  model cache: {ATST_MODEL_CACHE}")
print(f"dataset      : {DATASET_PATH}")
print(f"test audio   : {TEST_AUDIO_DIR}")
print(f"audiossl     : {HAS_ATST}")
print(f"ATST ckpt    : {ATST_FRAME_CKPT}  (exists: {os.path.isfile(ATST_FRAME_CKPT)})")
print(f"ATST-Frame   : {'yes' if HAS_ATST_FRAME else 'no -- falling back to AST'}")

## Cell 3 — Global config

Constants used throughout. `clip_sec` is auto-set to the max clip duration in the dataset in the next cell.

In [ ]:
# =============================================================================
# CELL 3 -- Global config (seeds, CFG dict, backbone selection, label space)
# =============================================================================

# ---- Seeds ----
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Main config ----
CFG = dict(
    sr=16000,               # sample rate the whole pipeline uses
    hop=160,                # 10 ms label raster hop
    clip_sec=5.0,           # overwritten in Cell 4 after data scan
    time_pool=2,            # 20 ms output step (2 × 10ms)
    val_frac=0.1,
    batch_size=4,           # reduced from 8 -- ATST-Frame is 4x longer seq than BEATs
    grad_accum_steps=4,     # compensate: effective batch stays 16
    stage1_epochs=3,
    stage2_epochs=8,
    lr_head=1e-3,
    lr_backbone=5e-5,
    weight_decay=1e-2,
    warmup_updates=200,
    early_stop_patience=3,
    num_workers=0,          # Windows spawn issues -- keep at 0
)

# ---- Backbone selection (defensive: works even if Cell 2 partially failed) ----
HAS_ATST_FRAME = globals().get("HAS_ATST_FRAME", False)
if HAS_ATST_FRAME:
    ATST_BACKBONE_NAME = "atst-frame"
    print(f"Backbone: ATST-Frame  (ckpt: {ATST_FRAME_CKPT})")
else:
    ATST_BACKBONE_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"
    print("Backbone: AST (HuggingFace fallback)")

# ---- Label space ----
# Track 1 scores only channel 0 ("any noise event"). The 7 category channels
# are auxiliary -- they give the model extra supervision signal but are not
# scored. NoiseCategory values in the Vaani dataset use exactly these 7 strings.
CATS = ["animal", "vehicle_traffic", "baby_child", "singing_music",
        "phone_signal_alarm", "appliance_machine", "human_non_speech"]
CAT2IDX = {c: i for i, c in enumerate(CATS)}
N_OUT = 1 + len(CATS)   # channel 0 = any-noise, channels 1..7 = categories

print(f"N_OUT={N_OUT}  CATS={CATS}")

## Cell 4 — Load local dataset (no HF Hub, no token)

Reads `DATASET_PATH` with `load_from_disk`. Decodes audio manually to avoid torchcodec issues.

In [ ]:
def decode_audio(a, target_sr=CFG['sr']):
    """Robust decoder for datasets.Audio payloads or filesystem paths."""
    w = sr = None
    if isinstance(a, dict):
        if a.get('array') is not None:
            w, sr = np.asarray(a['array'], dtype=np.float32), a['sampling_rate']
        elif a.get('bytes') is not None:
            try:
                w, sr = sf.read(io.BytesIO(a['bytes']), dtype='float32', always_2d=False)
            except Exception:
                with tempfile.NamedTemporaryFile(suffix='.audio', delete=True) as tmp:
                    tmp.write(a['bytes']); tmp.flush()
                    w, sr = librosa.load(tmp.name, sr=None, mono=True)
        elif a.get('path'):
            p = a['path']
            try:
                w, sr = sf.read(p, dtype='float32', always_2d=False)
            except Exception:
                w, sr = librosa.load(p, sr=None, mono=True)
    elif isinstance(a, (str, Path)) and str(a):
        p = str(a)
        try:
            w, sr = sf.read(p, dtype='float32', always_2d=False)
        except Exception:
            w, sr = librosa.load(p, sr=None, mono=True)
    if w is None:
        raise ValueError(f'cannot decode audio payload type={type(a)}')
    if w.ndim > 1:
        w = w.mean(axis=0) if w.shape[0] < w.shape[1] else w.mean(axis=1)
    if sr != target_sr:
        w = librosa.resample(w, orig_sr=sr, target_sr=target_sr)
    return np.ascontiguousarray(w, dtype=np.float32)


def spans_from(ex):
    """start/end/category per event, sorted by onset."""
    spans = []
    clip_cats = ex.get('NoiseCategory') or []
    fallback_cat = clip_cats[0] if clip_cats else None
    for s in (ex.get('NoiseSubCategoryTimeStamp') or []):
        try:
            st, en = float(s.get('start')), float(s.get('end'))
        except Exception:
            continue
        if en > st:
            spans.append((st, en, s.get('tag') or s.get('category') or fallback_cat))
    return sorted(spans)


def build_labels(spans, n_frames_100hz, has_strong=True):
    """(N_OUT, T_100hz) uint8 tensor. Channel 0 = any-noise, 1..7 = per-category."""
    y = np.zeros((N_OUT, n_frames_100hz), dtype=np.uint8)
    if has_strong:
        for st, en, cat in spans:
            a = max(0, int(round(st * 100)))
            b = min(n_frames_100hz, int(round(en * 100)))
            if b <= a:
                continue
            y[0, a:b] = 1
            ci = CAT2IDX.get(cat)
            if ci is not None:
                y[1 + ci, a:b] = 1
    return y


def load_local_rows(max_rows=None, log_every=2000):
    ds = load_from_disk(DATASET_PATH)
    n_total = len(ds)
    n_load = n_total if max_rows is None else min(max_rows, n_total)
    print(f"local dataset: {n_total} rows (loading {n_load})")
    rows, skipped = [], 0
    for i in range(n_load):
        ex = ds[i]
        q = ex.get('annotationQuality')
        has_strong = q in ('verified_timestamps', 'unverified_timestamps')
        try:
            wav = decode_audio(ex['audio'])
        except Exception as e:
            skipped += 1
            if skipped <= 10:
                print(f"  skip idx={i}: {type(e).__name__}: {e}")
            continue
        spans = spans_from(ex)
        nf = 1 + len(wav) // CFG['hop']
        lab = build_labels(spans, nf, has_strong=has_strong)
        rows.append({'wav': wav, 'lab': lab, 'quality': q, 'has_strong': has_strong})
        if (i + 1) % log_every == 0:
            print(f"  loaded {i+1}/{n_load} ({skipped} skipped)")
    if not rows:
        raise RuntimeError("No decodable rows. Check DATASET_PATH.")
    print(f"loaded {len(rows)} | skipped {skipped} | mix={Counter(r['quality'] for r in rows)}")
    return rows


rows = load_local_rows(max_rows=None)   # full dataset
dur = np.asarray([len(r['wav']) / CFG['sr'] for r in rows], dtype=np.float32)
print(f"duration: median={np.median(dur):.2f}s  p95={np.percentile(dur,95):.2f}s  max={dur.max():.2f}s")
CFG['clip_sec'] = float(math.ceil(dur.max()))
print(f"clip_sec auto-set to {CFG['clip_sec']}s")

N_SAMP     = int(CFG['clip_sec'] * CFG['sr'])
FRAMES_100 = int(N_SAMP / CFG['hop'])
FRAMES_OUT = FRAMES_100 // CFG['time_pool']
FRAME_SEC  = CFG['hop'] * CFG['time_pool'] / CFG['sr']
print(f"N_SAMP={N_SAMP}  FRAMES_OUT={FRAMES_OUT}  FRAME_SEC={FRAME_SEC:.4f}s")

## Cell 5 — Dataset, weights, loaders

Configures `Track1Dataset`, class-balanced positive weights, train/val split (clean-only val by default — the val clip must contain zero timestamped noise).

In [ ]:
USE_STRONG_ONLY = True     # exclude bronze (no timestamps) from frame-level training
VAL_CLEAN_ONLY  = True     # val split = clean clips only
QUALITY_WEIGHTS = {'verified_timestamps': 1.0,
                   'unverified_timestamps': 0.75,
                   'no_timestamps': 0.25}


def prep_wav(w):
    return (w - w.mean()) / (w.std() + 1e-5)


def is_clean_row(r):
    return float(np.asarray(r['lab'])[0].sum()) == 0.0


class Track1Dataset(Dataset):
    def __init__(self, rows, train=True):
        self.rows, self.train = rows, train
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        wav, lab = r['wav'], r['lab'].astype(np.float32)
        if len(wav) > N_SAMP:
            s = random.randint(0, len(wav) - N_SAMP) if self.train else 0
            wav = wav[s:s + N_SAMP]
            lab = lab[:, s // CFG['hop'] : s // CFG['hop'] + FRAMES_100]
            valid = N_SAMP
        else:
            valid = len(wav)
            wav = np.pad(wav, (0, N_SAMP - len(wav)))
        if lab.shape[1] < FRAMES_100:
            lab = np.pad(lab, ((0, 0), (0, FRAMES_100 - lab.shape[1])))
        lab = lab[:, :FRAMES_100]
        lab_20 = lab.reshape(lab.shape[0], FRAMES_OUT, CFG['time_pool']).max(axis=2)
        m = np.zeros(FRAMES_OUT, dtype=np.float32)
        n_valid = max(1, int(math.ceil(valid / CFG['sr'] / FRAME_SEC)))
        m[:min(n_valid, FRAMES_OUT)] = 1.0
        sw = float(QUALITY_WEIGHTS.get(r.get('quality'), 1.0))
        return {
            'wav':  torch.from_numpy(prep_wav(wav).astype(np.float32)),
            'lab':  torch.from_numpy(lab_20.astype(np.float32)),
            'mask': torch.from_numpy(m),
            'sw':   torch.tensor(sw, dtype=torch.float32),
        }


def estimate_pos_weight(rows):
    pos = np.zeros(N_OUT, dtype=np.float64)
    tot = np.zeros(N_OUT, dtype=np.float64)
    for r in rows:
        lab = r['lab'].astype(np.float32)
        if lab.shape[1] < FRAMES_100:
            lab = np.pad(lab, ((0, 0), (0, FRAMES_100 - lab.shape[1])))
        lab = lab[:, :FRAMES_100]
        lab_20 = lab.reshape(lab.shape[0], FRAMES_OUT, CFG['time_pool']).max(axis=2)
        nv = min(FRAMES_OUT, max(1, int(math.ceil(min(len(r['wav']), N_SAMP) / CFG['sr'] / FRAME_SEC))))
        y = lab_20[:, :nv]
        pos += y.sum(axis=1); tot += nv
    neg = np.maximum(tot - pos, 1.0)
    pw = np.clip(neg / np.maximum(pos, 1.0), 1.0, 50.0)
    return pw.astype(np.float32)


# Split
random.Random(SEED).shuffle(rows)
n_val = max(1, int(len(rows) * CFG['val_frac']))
val_rows, train_rows = rows[:n_val], rows[n_val:]

if USE_STRONG_ONLY:
    before = len(train_rows)
    train_rows = [r for r in train_rows if r['has_strong']]
    print(f"strong-only train: {before} -> {len(train_rows)}")

if VAL_CLEAN_ONLY:
    clean = [r for r in val_rows if is_clean_row(r)]
    if clean:
        val_rows = clean
        print(f"clean-only val: {len(val_rows)}")
    else:
        print("warning: clean-only val is empty; keeping mixed val")

POS_W = estimate_pos_weight(train_rows)
LOSS_POS_W = torch.tensor(POS_W, dtype=torch.float32, device=device)
print("channel pos_weight:", {i: round(float(w), 2) for i, w in enumerate(POS_W)})

train_dl = DataLoader(Track1Dataset(train_rows, True), batch_size=CFG['batch_size'],
                      shuffle=True, num_workers=CFG['num_workers'])
val_dl   = DataLoader(Track1Dataset(val_rows, False), batch_size=CFG['batch_size'],
                      shuffle=False, num_workers=CFG['num_workers'])
print(f"train clips={len(train_rows)}  val clips={len(val_rows)}")

In [ ]:
# Diagnostic: why is HAS_ATST_FRAME False?
import os

print("HAS_ATST          :", globals().get("HAS_ATST", "NOT DEFINED"))
print("HAS_ATST_FRAME    :", globals().get("HAS_ATST_FRAME", "NOT DEFINED"))
print("ATST_FRAME_CKPT   :", globals().get("ATST_FRAME_CKPT", "NOT DEFINED"))

ckpt = globals().get("ATST_FRAME_CKPT")
if ckpt:
    print("ckpt exists       :", os.path.isfile(ckpt))
    if os.path.isfile(ckpt):
        print("ckpt size (MB)    :", round(os.path.getsize(ckpt)/1e6, 1))

# Try the import directly
try:
    from audiossl.methods.atstframe.embedding import (
        load_model as load_atstframe,
        get_timestamp_embedding,
    )
    print("audiossl import   : OK")
except Exception as e:
    print(f"audiossl import   : FAILED -- {type(e).__name__}: {e}")

## Cell 6 — Model (ATST-Frame or AST backbone + BiGRU + frame/onset/offset heads)

Three heads: `strong` (frame presence), `onset` (boundary), `offset` (boundary). Clip-level attention pooling for weak supervision.

In [ ]:
# =============================================================================
# CELL 6 -- Model: ATST-Frame / AST backbone + BiGRU + per-frame heads
# =============================================================================
class ATSTSED(nn.Module):
    """ATST-Frame (or AST fallback) backbone + BiGRU + multi-head frame outputs."""

    def __init__(self, backbone_name, n_out=N_OUT, rnn_dim=256,
                 use_atst_frame=False, atst_ckpt=None):
        super().__init__()
        self.backbone_name = backbone_name
        self.use_atst_frame = use_atst_frame

        if use_atst_frame:
            # audiossl's load_model reconstructs the ATST-Frame Lightning module
            # from the raw ckpt (state_dict + hyper_parameters) and returns a
            # torch.nn.Module in eval mode.
            assert atst_ckpt is not None, "atst_ckpt path required when use_atst_frame=True"
            self.backbone = load_atstframe(atst_ckpt)
            self.backbone.eval()
            for p in self.backbone.parameters():
                p.requires_grad_(False)
            d = 768   # ATST-Frame base embedding dim (small variant would be 384)
            self.fe = None
            self.is_ast = False
            print(f"Backbone mode: ATST-Frame  (loaded from {atst_ckpt})")
        else:
            self.backbone = AutoModel.from_pretrained(backbone_name)
            self.model_type = getattr(self.backbone.config, 'model_type', '')
            self.is_ast = (self.model_type == 'audio-spectrogram-transformer')
            self.fe = (AutoFeatureExtractor.from_pretrained(backbone_name)
                       if self.is_ast else None)
            d = self.backbone.config.hidden_size
            print(f"Backbone mode: {self.model_type or 'waveform'}")

        self.rnn = nn.GRU(d, rnn_dim, 2, batch_first=True, bidirectional=True, dropout=0.1)
        self.head_frame  = nn.Linear(2 * rnn_dim, n_out)
        self.head_onset  = nn.Linear(2 * rnn_dim, n_out)
        self.head_offset = nn.Linear(2 * rnn_dim, n_out)
        self.att         = nn.Linear(2 * rnn_dim, n_out)

    def _encode(self, wav):
        """wav: (B, N_samples) float32 at 16 kHz. Returns (B, T_atst, 768)."""
        if self.use_atst_frame:
            with torch.no_grad():
                # get_timestamp_embedding handles fbank extraction, patch embed,
                # and transformer forward. Returns (emb, timestamps).
                emb, _ = get_timestamp_embedding(wav.detach(), self.backbone)
                if isinstance(emb, tuple):
                    emb = emb[0]
            return emb
        if self.is_ast:
            wav_np = wav.detach().float().cpu().numpy()
            feats = self.fe([w for w in wav_np], sampling_rate=CFG['sr'],
                            return_tensors='pt')
            inp = feats['input_values'].to(wav.device)
            out = self.backbone(inp)
        else:
            out = self.backbone(wav)
        return out.last_hidden_state

    def forward(self, wav, mask=None):
        h = self._encode(wav)                       # (B, T_atst, D)
        T = mask.shape[1] if mask is not None else h.shape[1]

        # Interpolate ATST's 40 ms frame grid (T_atst) to the label grid (T).
        if h.shape[1] != T:
            h = F.interpolate(h.transpose(1, 2), size=T, mode='linear',
                              align_corners=False).transpose(1, 2)

        h, _ = self.rnn(h)
        frame  = self.head_frame(h)
        onset  = self.head_onset(h)
        offset = self.head_offset(h)

        att = self.att(h)
        if mask is not None:
            att = att.masked_fill(mask.unsqueeze(-1) < 0.5, -1e4)
        att = torch.softmax(att, dim=1)
        clip = (torch.sigmoid(frame) * att).sum(dim=1).clamp(1e-6, 1 - 1e-6)

        return {
            'frame_logits':  frame.transpose(1, 2),    # (B, C, T)
            'onset_logits':  onset.transpose(1, 2),
            'offset_logits': offset.transpose(1, 2),
            'clip_prob':     clip,
        }


def bce_frame(logits, target, mask, sample_weight=None, pos_weight=None):
    """Class-balanced BCE on frame presence, masked to valid region."""
    if pos_weight is None:
        pos_weight = LOSS_POS_W if 'LOSS_POS_W' in globals() else None
    if pos_weight is None:
        pos_weight = torch.ones(logits.shape[1], device=logits.device, dtype=logits.dtype)
    pw = pos_weight.to(logits.device, dtype=logits.dtype).view(1, -1, 1)
    p = torch.sigmoid(logits).clamp(1e-6, 1 - 1e-6)
    loss = -(pw * target * torch.log(p) + (1 - target) * torch.log(1 - p))
    if sample_weight is not None:
        sw = sample_weight.to(target.device, dtype=target.dtype).view(-1, 1, 1)
        loss = loss * sw
    m = mask.unsqueeze(1).to(target.device, dtype=target.dtype)
    return (loss * m).sum() / (m.sum() * target.shape[1] + 1e-6)


# ---- Instantiate ----
model = ATSTSED(
    ATST_BACKBONE_NAME,
    n_out=N_OUT,
    rnn_dim=256,
    use_atst_frame=HAS_ATST_FRAME,
    atst_ckpt=ATST_FRAME_CKPT if HAS_ATST_FRAME else None,
).to(device)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
n_train  = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"params: {n_params:.1f}M total, {n_train:.1f}M trainable")

In [ ]:
# =============================================================================
# CELL: ATST-Frame embedding sanity check
# =============================================================================
import torch
from audiossl.methods.atstframe.embedding import load_model as _lm, get_timestamp_embedding as _gte

bb = _lm(ATST_FRAME_CKPT)
bb.eval()
x = torch.randn(2, 160000)   # batch of 2, 10s @ 16kHz
with torch.no_grad():
    emb, ts = _gte(x, bb)
if isinstance(emb, tuple): emb = emb[0]
print(f"emb shape : {tuple(emb.shape)}")    # expect (2, ~250, 768)
print(f"ts shape  : {tuple(ts.shape) if hasattr(ts, 'shape') else ts}")
print(f"emb mean  : {emb.mean().item():.4f}")
print(f"emb std   : {emb.std().item():.4f}")
print(f"isnan     : {torch.isnan(emb).any().item()}")
del bb
torch.cuda.empty_cache()

## Cell 7 — Training (two stages, checkpoints + logs to disk)

Stage 1: freeze backbone, train heads.
Stage 2: unfreeze backbone at low LR.
Progress logged every epoch to `progress_log.txt` (append) and every 100 updates to `progress_live.txt` (overwrite).

In [ ]:
from tqdm.auto import tqdm

scaler = GradScaler(enabled=torch.cuda.is_available())


def _write_log(line, path=ATST_PROGRESS_LOG):
    print(line, flush=True)
    try:
        with open(path, "a", encoding="utf-8") as f:
            f.write(line + "\n")
    except Exception as e:
        print(f"[warn] log write failed: {e}")


def _write_live(text, path=ATST_PROGRESS_LIVE):
    try:
        with open(path, "w", encoding="utf-8") as f:
            f.write(text)
    except Exception:
        pass


def eval_loss(model, dl, desc="val"):
    model.eval()
    vals = []
    with torch.no_grad():
        for batch in tqdm(dl, desc=desc, leave=False):
            wav = batch['wav'].to(device, non_blocking=True)
            lab = batch['lab'].to(device, non_blocking=True)
            m   = batch['mask'].to(device, non_blocking=True)
            sw  = batch.get('sw', None)
            if sw is not None:
                sw = sw.to(device, non_blocking=True)
            with autocast('cuda', enabled=torch.cuda.is_available()):
                out = model(wav, m)
                loss = bce_frame(out['frame_logits'].float(), lab.float(), m.float(), sample_weight=sw)
            vals.append(float(loss.detach().cpu()))
    return float(np.mean(vals)) if vals else 0.0


def _train_stage(stage_name, n_epochs, optimizer, params_for_clip):
    prev = None
    inc = 0
    for ep in range(1, n_epochs + 1):
        model.train()
        tr = []
        pbar = tqdm(train_dl, desc=f"{stage_name} ep{ep:02d}", leave=False)
        for bi, batch in enumerate(pbar):
            wav = batch['wav'].to(device, non_blocking=True)
            lab = batch['lab'].to(device, non_blocking=True)
            m   = batch['mask'].to(device, non_blocking=True)
            sw  = batch.get('sw', None)
            if sw is not None:
                sw = sw.to(device, non_blocking=True)
            with autocast('cuda', enabled=torch.cuda.is_available()):
                out = model(wav, m)
                loss = bce_frame(out['frame_logits'].float(), lab.float(), m.float(), sample_weight=sw) \
                       / CFG['grad_accum_steps']
            scaler.scale(loss).backward()
            if (bi + 1) % CFG['grad_accum_steps'] == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(params_for_clip, 5.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            tr.append(float(loss.detach().cpu()) * CFG['grad_accum_steps'])
            pbar.set_postfix(loss=f"{tr[-1]:.4f}")
            if (bi + 1) % 100 == 0:
                _write_live(
                    f"stage={stage_name}\nepoch={ep}/{n_epochs}\n"
                    f"step={bi+1}/{len(train_dl)}\nloss={tr[-1]:.4f}\n"
                    f"time={time.strftime('%H:%M:%S')}\n")
        tr_mean = float(np.mean(tr))
        vl = eval_loss(model, val_dl, desc=f"{stage_name} ep{ep:02d} val")
        line = (f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {stage_name} "
                f"epoch {ep}/{n_epochs} train={tr_mean:.4f} val={vl:.4f}")
        _write_log(line)
        ck = f"{ATST_CKPT_DIR}/best_{stage_name}.pt"
        if prev is None or vl < prev:
            torch.save({'model': model.state_dict(), 'cfg': CFG,
                        'backbone': ATST_BACKBONE_NAME, 'val_loss': vl,
                        'stage': stage_name, 'epoch': ep}, ck)
            _write_log(f"  saved {ck} (val={vl:.4f})")
        if prev is not None and (vl - prev) > 1e-6:
            inc += 1
        else:
            inc = 0
        prev = vl
        if inc >= CFG['early_stop_patience']:
            _write_log(f"  early stop: val worsened {inc} epochs")
            return
    return


# ---- Stage 1: frozen backbone, head only ----
if HAS_ATST_FRAME:
    backbone_params = model.backbone.parameters()
else:
    backbone_params = model.backbone.parameters()

for p in backbone_params:
    p.requires_grad_(False)
head_params = [p for p in model.parameters() if p.requires_grad]
opt1 = torch.optim.AdamW(head_params, lr=CFG['lr_head'], weight_decay=CFG['weight_decay'])
_write_log(f"\n=== STAGE 1 ({CFG['stage1_epochs']} epochs, frozen backbone) ===")
_write_log(f"trainable params: {sum(p.numel() for p in head_params):,}")
_train_stage("stage1", CFG['stage1_epochs'], opt1, head_params)

# ---- Stage 2: unfreeze backbone at low LR ----
for p in backbone_params:
    p.requires_grad_(True)
opt2 = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': CFG['lr_backbone']},
    {'params': [p for n, p in model.named_parameters() if not n.startswith('backbone.')],
     'lr': CFG['lr_head'] * 0.3},
], weight_decay=CFG['weight_decay'])
_write_log(f"\n=== STAGE 2 ({CFG['stage2_epochs']} epochs, unfrozen @ lr={CFG['lr_backbone']}) ===")
_train_stage("stage2", CFG['stage2_epochs'], opt2, list(model.parameters()))

_write_log("\n=== TRAINING DONE ===")

## Cell 8 — Inference helpers (frame probs -> events)

Produces tracks (per-frame frame/onset/offset probs) for any clip, then decodes events via threshold + median filter + merge.

In [ ]:
def prob_to_events(p, thr=0.5, med=7, frame_sec=FRAME_SEC, min_dur=0.05, merge_gap=0.10):
    b = (np.asarray(p) >= thr).astype(np.uint8)
    if med > 1:
        b = median_filter(b, size=med, mode='nearest')
    events, start = [], None
    for i, v in enumerate(b):
        if v and start is None:
            start = i
        elif (not v) and start is not None:
            events.append([start * frame_sec, i * frame_sec])
            start = None
    if start is not None:
        events.append([start * frame_sec, len(b) * frame_sec])
    out = []
    for e in events:
        if out and e[0] - out[-1][1] <= merge_gap:
            out[-1][1] = e[1]
        else:
            out.append(e)
    return [e for e in out if e[1] - e[0] >= min_dur]


@torch.no_grad()
def _infer_chunk(wav_chunk):
    model.eval()
    x = torch.from_numpy(prep_wav(wav_chunk)).unsqueeze(0).to(device)
    m = torch.ones(1, FRAMES_OUT, device=device)
    out = model(x, m)
    return {
        'frame':  torch.sigmoid(out['frame_logits'][0, 0]).float().cpu().numpy(),
        'onset':  torch.sigmoid(out['onset_logits'][0, 0]).float().cpu().numpy(),
        'offset': torch.sigmoid(out['offset_logits'][0, 0]).float().cpu().numpy(),
        'clip_prob': float(torch.sigmoid(out['clip_prob'][0, 0]).cpu()),
    }


@torch.no_grad()
def infer_tracks(wav, overlap=0.5):
    """Full-clip frame/onset/offset probability tracks (overlap-add for long clips)."""
    wav = wav.astype(np.float32)
    if len(wav) <= N_SAMP:
        padded = np.pad(wav, (0, N_SAMP - len(wav)))
        return _infer_chunk(padded)
    frame_samp = CFG['hop'] * CFG['time_pool']
    total_frames = int(math.ceil(len(wav) / frame_samp))
    acc = {k: np.zeros(total_frames, dtype=np.float32) for k in ('frame', 'onset', 'offset')}
    cnt = np.zeros(total_frames, dtype=np.float32)
    clip_probs = []
    step = max(frame_samp, int(N_SAMP * (1 - overlap)))
    starts = list(range(0, max(1, len(wav) - N_SAMP + 1), step))
    if starts[-1] != len(wav) - N_SAMP:
        starts.append(max(0, len(wav) - N_SAMP))
    for s in starts:
        ch = wav[s:s + N_SAMP]
        if len(ch) < N_SAMP:
            ch = np.pad(ch, (0, N_SAMP - len(ch)))
        tr = _infer_chunk(ch)
        g0 = int(round(s / frame_samp))
        g1 = min(total_frames, g0 + len(tr['frame']))
        L = max(0, g1 - g0)
        if L > 0:
            for k in ('frame', 'onset', 'offset'):
                acc[k][g0:g1] += tr[k][:L]
            cnt[g0:g1] += 1.0
        clip_probs.append(tr['clip_prob'])
    out = {k: acc[k] / np.maximum(cnt, 1e-6) for k in acc}
    out['clip_prob'] = float(np.mean(clip_probs)) if clip_probs else 1.0
    return out


def predict_events(wav, thr=0.5, med=7):
    tr = infer_tracks(wav)
    events = prob_to_events(tr['frame'], thr=thr, med=med)
    return [{'onset': round(float(a), 3), 'offset': round(float(b), 3)} for a, b in events]

## Cell 9 — Decoder threshold tuning on val

Sweeps `thr` and `med` over a small grid, picks the pair maximizing Event-F1 on val.

In [ ]:
import pickle
from scipy.ndimage import median_filter as mf

# Cache val tracks once (used for tuning + reference metrics)
print("computing val tracks...")
val_cache = {}
t0 = time.time()
for i, r in enumerate(tqdm(val_rows, desc="val")):
    tr = infer_tracks(r['wav'])
    val_cache[i] = {'tracks': tr, 'lab': r['lab'].astype(np.float32), 'wav_len': len(r['wav'])}
print(f"val tracks: {len(val_cache)} clips in {time.time()-t0:.1f}s")


def _dice(p_any, y_any):
    p = np.asarray(p_any) >= 0.5
    y = np.asarray(y_any) >= 0.5
    inter = float((p & y).sum())
    tot = float(p.sum() + y.sum())
    return 0.0 if tot == 0 else 2.0 * inter / tot


def _event_match(g, p):
    dur = max(1e-6, g[1] - g[0])
    tol = max(0.2 * dur, 0.05)
    return (abs(g[0] - p[0]) <= tol) and (abs(g[1] - p[1]) <= tol)


def score_val(thr=0.5, med=7):
    tp = fp = fn = 0
    dices = []
    for i, v in val_cache.items():
        tr = v['tracks']
        y = v['lab'][0]  # channel 0 = any-noise
        # ground truth events
        gt_events = []
        in_ev = False; st = 0
        for t_, val in enumerate(y):
            if val and not in_ev:
                in_ev, st = True, t_
            elif (not val) and in_ev:
                in_ev = False
                gt_events.append((st * FRAME_SEC, t_ * FRAME_SEC))
        if in_ev:
            gt_events.append((st * FRAME_SEC, len(y) * FRAME_SEC))
        pred = prob_to_events(tr['frame'], thr=thr, med=med)
        # F1
        used = set()
        for g in gt_events:
            best = -1; best_err = 1e9
            for j, p in enumerate(pred):
                if j in used: continue
                if _event_match(g, p):
                    err = abs(g[0]-p[0]) + abs(g[1]-p[1])
                    if err < best_err:
                        best_err, best = err, j
            if best >= 0:
                used.add(best); tp += 1
        fp += len(pred) - tp
        fn += len(gt_events) - tp
        dices.append(_dice(tr['frame'] >= 0.5, y))
    p = tp / (tp + fp + 1e-9); r = tp / (tp + fn + 1e-9)
    f1 = 2 * p * r / (p + r + 1e-9)
    dice = float(np.mean(dices)) if dices else 0.0
    return dict(f1=f1, dice=dice, combined=f1 + dice)


print("\nthreshold sweep:")
best = None
for thr in (0.3, 0.4, 0.5, 0.6, 0.7):
    for med in (1, 3, 5, 7, 11):
        sc = score_val(thr=thr, med=med)
        tag = f"thr={thr:.1f} med={med:2d}"
        print(f"  {tag}: F1={sc['f1']:.4f} Dice={sc['dice']:.4f} Combined={sc['combined']:.4f}")
        if best is None or sc['combined'] > best[2]['combined']:
            best = (thr, med, sc)

BEST_THR, BEST_MED, BEST_SC = best
print(f"\nBEST: thr={BEST_THR} med={BEST_MED} -> Combined={BEST_SC['combined']:.4f}")
_write_log(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] DECODER tuned thr={BEST_THR} med={BEST_MED} "
           f"F1={BEST_SC['f1']:.4f} Dice={BEST_SC['dice']:.4f} Combined={BEST_SC['combined']:.4f}")

## Cell 10 — Test inference + submission

Runs inference on all audio in `TEST_AUDIO_DIR` and writes `submission.zip`.

In [ ]:
AUDIO_EXTS = ('.wav', '.flac', '.mp3', '.ogg', '.m4a')
test_files = sorted(p for p in Path(TEST_AUDIO_DIR).rglob('*')
                    if p.suffix.lower() in AUDIO_EXTS)
print(f"found {len(test_files)} test clips")


def _read_test_audio(p):
    w, sr = sf.read(str(p), dtype='float32', always_2d=False)
    if w.ndim > 1:
        w = w.mean(axis=1)
    if sr != CFG['sr']:
        w = librosa.resample(w, orig_sr=sr, target_sr=CFG['sr'])
    return w.astype(np.float32)


predictions = {}
t0 = time.time()
for i, p in enumerate(tqdm(test_files, desc="test")):
    cid = p.stem
    wav = _read_test_audio(p)
    predictions[cid] = predict_events(wav, thr=BEST_THR, med=BEST_MED)
    if (i + 1) % 500 == 0:
        print(f"  [{i+1}/{len(test_files)}] elapsed={time.time()-t0:.0f}s")

n_empty = sum(1 for v in predictions.values() if not v)
n_ev = sum(len(v) for v in predictions.values())
print(f"\ntest done: {len(predictions)} clips, {n_empty} empty, {n_ev} events")

# Write submission
out_jsonl = f"{ATST_SUB_DIR}/predictions.jsonl"
out_zip   = f"{ATST_SUB_DIR}/submission.zip"
with open(out_jsonl, "w", encoding="utf-8") as f:
    for cid, evs in predictions.items():
        f.write(json.dumps({"clip_id": cid, "events": evs}, ensure_ascii=False) + "\n")
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(out_jsonl, "predictions.jsonl")
print(f"\nwrote {out_zip}")

## Cell 11 — Resume / sanity checks

Utility cells for debugging. Run only if needed.

In [ ]:
# Checkpoint inspection
import glob
for ck in sorted(glob.glob(f"{ATST_CKPT_DIR}/*.pt")):
    d = torch.load(ck, map_location='cpu', weights_only=False)
    print(f"{Path(ck).name}: stage={d.get('stage')} epoch={d.get('epoch')} val_loss={d.get('val_loss'):.4f}")

# Log tail
print("\n--- progress_log.txt (last 10) ---")
try:
    with open(ATST_PROGRESS_LOG) as f:
        lines = f.readlines()[-10:]
        print("".join(lines))
except FileNotFoundError:
    print("(no log yet)")